# Fusion finale — pollution horaire + features journalières

**Objectif** : construire le dataset d'entraînement complet du modèle XGBoost en
broadcastant les features journalières (météo + événements + validations) sur les
70 272 lignes horaires de pollution.

**Opération** : un `merge` sur la clé `DATE`. La granularité horaire de la pollution
est **strictement préservée** — on ne fait qu'ajouter des colonnes à droite. Chaque
valeur journalière sera dupliquée 192 fois (24 heures × 8 segments) dans le résultat,
ce qui est l'effet attendu du broadcast.

**Résultat attendu** : 70 272 lignes × ~46 colonnes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data/processed")
OUT_PATH = DATA_DIR / "dataset_entrainement_2024.csv"


## Étape 1 — Chargement des deux briques

In [ ]:
pollution  = pd.read_csv(DATA_DIR / "pollution_processed_2024.csv")
journalier = pd.read_csv(DATA_DIR / "dataset_journalier_2024.csv")

print(f"Pollution  : {pollution.shape}   (granularité : heure × segment)")
print(f"Journalier : {journalier.shape}    (granularité : jour)")
print(f"\nColonnes journalier qui vont être broadcastées :")
print(f"  {[c for c in journalier.columns if c != 'DATE']}")

## Étape 2 — Merge sur DATE

Deux arguments clés :
- **`how="left"`** : on garde toutes les lignes de pollution, même si une date manquait
  dans le journalier (ne devrait pas arriver, mais c'est sécurisant).
- **`validate="many_to_one"`** : chaque date dans `journalier` doit être unique. Si
  jamais il y avait un doublon (deux lignes pour le 2024-03-15), pandas planterait au
  lieu de produire un résultat silencieusement faux.

In [ ]:
final = pollution.merge(
    journalier,
    on="DATE",
    how="left",
    validate="many_to_one",
)

print(f"Dataset final : {final.shape}")
print(f"\nNombre de lignes inchangé par rapport à pollution : {final.shape[0] == pollution.shape[0]}")

## Étape 3 — Sanity checks

On vérifie trois choses :

1. **La pollution est intacte** : les colonnes NO2/PM10/PM25 du `final` doivent être
   strictement identiques à celles de `pollution` (le broadcast ne touche jamais aux
   données horaires).
2. **NaN sous contrôle** : les seuls NaN attendus sont dans `NOM_FERIE` et
   `TYPE_VACANCES`, hérités du journalier (jours non fériés / hors vacances).
3. **Le broadcast fonctionne** : pour un jour donné, chaque feature journalière doit
   être identique sur les 192 lignes (24 h × 8 segments).

In [ ]:
# 1. Pollution intacte
for col in ["NO2", "PM10", "PM25"]:
    identique = (final[col].values == pollution[col].values).all()
    print(f"  {col} inchangé : {identique}")

# 2. NaN
print(f"\nNaN totaux : {final.isna().sum().sum():,}")
print(f"NaN par colonne (uniquement celles avec NaN) :")
print(final.isna().sum()[final.isna().sum() > 0])

In [ ]:
# 3. Broadcast — vérifier qu'un jour donné a bien la même valeur journalière partout
date_test = "2024-03-15"
echantillon = final[final["DATE"] == date_test]
print(f"Lignes pour le {date_test} : {len(echantillon)}  (attendu : 24 × 8 = 192)")

for col in ["MAX_TEMPERATURE_C", "VALD_NAVIGO", "JOUR_FERIE"]:
    n_unique = echantillon[col].nunique()
    val = echantillon[col].iloc[0]
    print(f"  {col} : {n_unique} valeur unique = {val}  {'✓' if n_unique == 1 else '✗'}")

## Étape 4 — Réorganisation des colonnes

Ordre logique pour la lisibilité : clés → cibles → features temps fin → calendrier →
météo → transport.

In [ ]:
cles      = ["time", "DATE", "segment"]
cibles    = ["NO2", "PM10", "PM25"]
temps_fin = ["HEURE", "HEURE_SIN", "HEURE_COS", "PERIODE"]
calendrier = ["JOUR_SEMAINE", "NOM_JOUR", "WEEKEND", "MOIS", "NUM_SEMAINE", "JOUR_ANNEE",
              "JOUR_FERIE", "NOM_FERIE", "VACANCES_SCOLAIRES", "TYPE_VACANCES",
              "JO", "JOP", "EVENEMENT_OLYMPIQUE", "JOUR_NON_OUVRE", "JOUR_PERTURBE"]
meteo     = ["MAX_TEMPERATURE_C", "MIN_TEMPERATURE_C", "TEMP_AVG_C", "TEMP_RANGE_C",
             "WINDSPEED_MAX_KMH", "PRECIP_TOTAL_DAY_MM", "HUMIDITY_MAX_PERCENT",
             "VISIBILITY_AVG_KM", "PRESSURE_MAX_MB", "CLOUDCOVER_AVG_PERCENT",
             "UV_INDEX", "SUNHOUR", "TOTAL_SNOW_MM"]
transport = ["VALD_NAVIGO", "VALD_IMAGINE_R", "VALD_SOLIDARITE", "VALD_AUTRES",
             "VALD_AMETHYSTE", "VALD_COURTS", "VALD_NON_DEFINI", "VALD_TOTAL"]

ordre = cles + cibles + temps_fin + calendrier + meteo + transport

# Sécurité : ne garder que les colonnes qui existent vraiment
ordre = [c for c in ordre if c in final.columns]
final = final[ordre]

print(f"Colonnes réordonnées : {len(final.columns)} colonnes")
print("\nAperçu :")
final.head(3)

## Étape 5 — Sauvegarde

In [ ]:
final.to_csv(OUT_PATH, index=False)

size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"✓ Sauvegardé : {OUT_PATH}")
print(f"  {final.shape[0]:,} lignes × {final.shape[1]} colonnes  ({size_mb:.1f} Mo)")

---

**Sortie** : `data/processed/dataset_entrainement_2024.csv`

Prochaine étape : modélisation XGBoost.
- Split train/test temporel (janv-oct / nov-déc)
- Encodage des catégorielles (`segment`, `PERIODE`, `NOM_FERIE`, `TYPE_VACANCES`)
- Choix de la cible : `NO2`
- Évaluation : RMSE, MAE, R² + feature importance